# Voice Clone Google Colab GPU Backend

Notebook này cho phép khởi chạy Backend của ứng dụng **Voice Clone** trên GPU miễn phí của Google Colab.
Rất hữu ích khi bạn sử dụng máy tính cá nhân cấu hình thấp (không có GPU NVIDIA hoặc ít RAM).

### Hướng dẫn sử dụng:
1. Truy cập menu **Runtime** -> **Change runtime type** -> Chọn GPU làm Hardware accelerator.
2. Nhấn nút **Chạy tất cả (Run all)** hoặc chạy từng ô code bên dưới theo thứ tự.
3. Chờ cho đến khi Cloudflare Tunnel khởi tạo xong, hệ thống sẽ in ra một liên kết dạng `https://xxx.trycloudflare.com`.
4. Sao chép địa chỉ đó, mở ứng dụng **Voice Clone Desktop**, chuyển sang chế độ **Google Colab GPU** và dán địa chỉ đó vào cấu hình.

### Tính năng Whisper Transcribe (Tạo SRT):
Bạn có thể gửi API POST đến `{URL}/transcribe` để tạo phụ đề (SRT) từ file audio.
Để kiểm tra trạng thái và lấy thông tin, gọi GET `{URL}/status/{task_id}`.
Để tải file SRT, gọi GET `{URL}/download/{task_id}`.

In [ ]:
#@title 1. Cài đặt các gói thư viện phụ thuộc (Dependencies)
import os
import sys

print("--- 1. Đang tải mã nguồn ứng dụng từ GitHub... ---")
if not os.path.exists("/content/voicecolab"):
    !git clone https://github.com/nqthaivl/voicecolab.git /content/voicecolab

%cd /content/voicecolab

print("\n--- 2. Đang cài đặt các thư viện cần thiết... ---")
!pip install -r backend/requirements.txt
# Cài đặt thêm huggingface_hub
!pip install huggingface_hub

print("\n--- 3. Tải xuống Cloudflare Tunnel để tạo đường truyền kết nối công khai... ---")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("\nHoàn tất bước chuẩn bị môi trường!")

In [ ]:
#@title 2. Tải trước mô hình AI
from huggingface_hub import snapshot_download
import json
from pathlib import Path

model_dir = Path("/content/data/models/omnivoice")
model_id = "k2-fsa/OmniVoice"
marker_path = model_dir / ".voice-clone-model.json"

if not marker_path.is_file():
    print("Đang tải mô hình từ HuggingFace về bộ nhớ đệm Colab (khoảng 1.5GB - 2GB)...")
    model_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=model_id,
        local_dir=str(model_dir),
        max_workers=4
    )
    marker_path.write_text(json.dumps({"model_id": model_id}, ensure_ascii=False), encoding="utf-8")
    print("\nTải mô hình thành công!")
else:
    print("Mô hình đã được tải và sẵn sàng sử dụng.")

In [ ]:
#@title 3. Khởi chạy Backend và tạo đường truyền liên kết kết nối
import subprocess
import time
import re
import sys

# Đảm bảo tạo sẵn thư mục lưu trữ data
data_dir = "/content/data"
os.makedirs(os.path.join(data_dir, "voices"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "outputs"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "temp"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "models"), exist_ok=True)

# Khởi động FastAPI Backend (uvicorn)
print("Đang khởi động backend Voice Clone...")
backend_process = subprocess.Popen(
    ["python", "run.py"],
    cwd="/content/voicecolab/backend",
    env={
        **os.environ,
        "VOICE_CLONE_PORT": "3920",
        "VOICE_CLONE_DATA_DIR": data_dir
    }
)

# Đợi backend uvicorn chạy (khoảng 3 giây)
time.sleep(3)

# Chạy Cloudflare Tunnel
print("Đang khởi tạo Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:3920"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

colab_url = None
try:
    while True:
        line = tunnel_process.stdout.readline()
        if not line:
            break
        # In ra các thông tin log cần thiết
        if "trycloudflare.com" in line or "error" in line.lower() or "tunnel" in line.lower():
            print("[Cloudflared]", line.strip())
        
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            colab_url = match.group(0)
            print("\n" + "="*60)
            print(f"🎉 KHỞI CHẠY GOOGLE COLAB BACKEND THÀNH CÔNG!")
            print(f"🔗 Địa chỉ API Colab của bạn là:")
            print(f"   {colab_url}")
            print("\nHướng dẫn tiếp theo:")
            print("1. Sao chép địa chỉ trên (bắt đầu bằng https)")
            print("2. Mở ứng dụng Voice Clone Desktop -> chọn tab Cấu hình (Settings)")
            print("3. Chọn chế độ: Google Colab GPU")
            print("4. Dán địa chỉ vào ô 'URL Google Colab' rồi nhấn 'Lưu cấu hình'")
            print("\n============================================================")
            print("📌 API Whisper Transcribe:")
            print(f"   - Tạo SRT : POST {colab_url}/transcribe")
            print(f"   - Status  : GET {colab_url}/status/{{task_id}}")
            print(f"   - Tải SRT : GET {colab_url}/download/{{task_id}}")
            print("="*60 + "\n")
            
    # Giữ cho tiến trình chạy để tiếp tục phục vụ
    backend_process.wait()
except KeyboardInterrupt:
    print("\nĐang dừng các tiến trình...")
    tunnel_process.terminate()
    backend_process.terminate()
    print("Đã dừng.")
